# Sessao Spark para uso do delta sharing

## Usando spark-submit
Se você estiver executando seu código como um aplicativo com spark-submit, você pode incluir o pacote usando a flag --packages:

```shell
spark-submit --packages io.delta:delta-sharing-spark_2.12:0.7.0 seu_script.py
```

## Usando pyspark ou Jupyter/Databricks
Se você estiver usando pyspark interativamente ou em um notebook (como Jupyter), você pode configurar isso ao criar sua SparkSession:

```python
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DeltaSharingApp") \
    .config("spark.jars.packages", "io.delta:delta-sharing-spark_2.12:0.7.0") \
    .getOrCreate()

df = spark.read.format("deltaSharing").load(table_url)
```

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import delta_sharing

In [2]:
spark = (
    SparkSession
    .builder.appName("DeltaSharingApp")
    .config("spark.jars.packages",
            "org.mongodb.spark:mongo-spark-connector_2.12:3.0.1,"
            "io.delta:delta-sharing-spark_2.12:0.7.0")
    .getOrCreate()
)

In [3]:
profile_file = "./config.share"
client = delta_sharing.SharingClient(profile_file)

In [4]:
client.list_all_tables()

[Table(name='contact', share='core_share', schema='deltashare_core'),
 Table(name='broad_notifications_dataflow', share='core_share', schema='deltashare_core'),
 Table(name='messages', share='core_share', schema='deltashare_core'),
 Table(name='notifications', share='core_share', schema='deltashare_core'),
 Table(name='tickets_new', share='core_share', schema='deltashare_core'),
 Table(name='eventtracks', share='core_share', schema='deltashare_core')]

In [5]:
share_name = "core_share"
schema_name = "deltashare_core"

In [8]:
#table_url = profile_file + "#<share-name>.<schema-name>.<table-name>"
table_url = profile_file + f"#{share_name}.{schema_name}.messages"

In [9]:
df_streaming = (
    spark.readStream
    .format("deltaSharing")
    .option("ignoreDeletes", "true")
    .load(table_url)
)

# df_streaming = df_streaming.dropDuplicates()

In [10]:
query = (
    df_streaming.writeStream
    .outputMode("append")
    .format("parquet")
    .option("path", "streaming/data")
    # .option("path", "streaming/data_2_dropDuplicates")
    .option("maxRecordsPerFile", 1000)
    .partitionBy('StorageDateDayBR')
    .trigger(processingTime="5 minutes")
    .option("checkpointLocation", "streaming/checkpoint")
    # .option("checkpointLocation", "streaming/checkpoint_2_dropDuplicates")
    .start()
)

query.awaitTermination()

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 